## 1. Clean Quarterly DCRA Data

The first step in integrating the Death in Custody Act (DCRA) quarterly reports to TJI's data is understanding, cleaning, and converting the data into a standardized format. 

The target output of this task will be a function with the following specifications:
- Input: S3 uri (example: `s3://arnold-ventures-project/dcra/raw_reports/475-23 Moravec - Documents for Release_16.xlsx`)
- Output: The resulting data is loaded into a SQL table in our data warehouse (`adhoc.dcra_reports`)

The final dataset will have:
- All column headers in [snake case](https://en.wikipedia.org/wiki/Snake_case)
- A data dictionary with all column meanings/any transformations performed
- All true/false columns converted to boolean datatype (for example, convert any column where X is true and null is false to explicitly use true and false)
- Any placeholder numbers (eg, 0 or 9999 for nulls) are replaced with actual null values
- Any other transformations as needed 

Links:
- [All datasets (TJI google drive)](https://drive.google.com/drive/u/0/folders/1Oci6fIlZC5_FA1lZwOkGZGAn6Ym6YAyH)
- [Reporting FAQ](https://drive.google.com/file/d/1Ve5-HPKZ0VV4taOuuO8p-HvgVOYCyoVU/view?usp=drive_link)
- [Questionaire used to create dataset](https://drive.google.com/file/d/1jFdcn4YvuHE9QaAA-QQNmpaxLIBm-mGq/view?usp=drive_link)

In [51]:
import pandas as pd
import dotenv
import sqlalchemy as sa
import os

In [52]:
dotenv.load_dotenv(f"{os.environ['HOME']}/.env")
postgres_engine = sa.create_engine(
    f'postgresql+psycopg2://{os.getenv("DB_USERNAME")}:{os.getenv("DB_PASSWORD")}@{os.getenv("DB_URI")}:{os.getenv("DB_PORT")}/{os.getenv("DB_NAME")}'
)

# example code to write to database
# with postgres_engine.begin() as con:
#     final_df.to_sql(name='dcra_reports', schema='adhoc', index=False, con=con, if_exists='append')

In [53]:
S3_URI = 's3://arnold-ventures-project/dcra/raw_reports/475-23 Moravec - Documents for Release_16.xlsx'
df = pd.read_excel(S3_URI)

In [54]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [55]:
rename_dict = {'gender_-_1._male': 'gender_male', 
               'gender_-_2._female': 'gender_female', 
               'gender_-_3._other': 'gender_other', 
               'please_specify_other_gender_identity': 'gender_other_desc', 
               'race_(select_all_that_apply)_-_1._american_indian_or_alaska_native': 'american_indian_or_alaska_native', 
               'race_(select_all_that_apply)_-_2._asian': 'asian', 
               'race_(select_all_that_apply)_-_3._black/african_american': 'black_african_american', 
               'race_(select_all_that_apply)_-_4.__native_hawaiian_or_other_pacific_islander': 'native_hawaiian_other_pacific_islander',
               'race_(select_all_that_apply)_-_5._white_': 'white', 
               'race_-_6._unknown': 'unknown', 
               'd._ethnicity_-_1._hispanic,_latino,_or_spanish_origin': 'hispanic', 
               '2._not_of_hispanic,_latino,_or_spanish_origin': 'not_hispanic', 
               '3._unknown': 'unknown', 
               'birth_year_(yyyy)_if_unknown,_please_enter_"9999"': 'birth_year', 
               'a._date_of_death_(mm-dd-yyyy)': 'death_date', 
               'b._time_of_death_(24-hour_clock_hh:mm)': 'death_time', 
               'location_of_death:_1._location_name': 'death_location', 
               '2._street_address': 'death_street_address', 
               '3._city': 'death_city', 
               '4._state': 'death_state', 
               '5._zip': 'death_zip', 
               'if_the_event_causing_the_death_occurred_in_any_of_the_following_facilities:__1.__municipal_or_county_jail': 'municipal_or_county_jail', 
               '2.__state_prison': 'state_prison', 
               '3.__state-run_boot_camp_prison': 'state_run_boot_camp_prison', 
               '4.__contracted_boot_camp_prison': 'contracted_boot_camp_prison', 
               '5.__any_state_or_local_contract_facility': 'any_state_or_local_contract_facility', 
               '6.__other_local_or_state_correctional_facility_(to_include_any_juvenile_facilities)': 'other_local_or_state_correctional_facility', 
               '7.__none_of_the_above': 'none_of_the_above', 
               '_please_list_the_name_of_the_department_or_agency_that_detained,_arrested,_or_was_in_the_process_of_arresting_the_deceased._(agency_name)': 'agency_name', 
               'please_indicate_the_manner_of_death_(mark_only_one)_________________________________________________________________________________a.__execution': 'execution', 
               'please_indicate_the_manner_of_death_(mark_only_one)_________________________________________________________________________________b.__accident': 'accident', 
               'c.__death_attributed_to_use_of_force_by_a_law_enforcement_or_corrections_officer': 'use_of_force', 
               'd__homicide_(e.g.,_an_incident_between_two_or_more_incarcerated_individuals_resulting_in_a_death)': 'homicide', 
               'e.__natural_causes': 'natural_causes', 
               'f.__suicide': 'suicide', 
               'g.__unavailable,_investigation_pending': 'unavailable_investigation_pending', 
               'if_yes,_please_report_the_agency_conducting_the_investigation_and_an_approximate_end_date._when_the_investigation_has_concluded,_please_contact_the_pmt_help_desk_to_update_this_report.:': 'investigation_pending_desc', 
               'h.__other': 'other', 
               'if_other,_please_explain:': 'other_desc', 
               'please_provide_a_brief_description_of_the_circumstances_leading_to_the_death_(e.g.,_details_surrounding_an_event_that_may_have_led_to_the_death,_the_number_and_affiliation_of_any_parties_involved_in_an_incident,_the_location_and_characteristics_of_an_incident,_other_context_related_to_the_death,_etc.).': 'death_description'}

In [56]:
df.rename(columns=rename_dict, inplace=True)

In [57]:
# check gender columns and combine into one column 
def gender(row):
    if row['gender_male']:
        return 'male'
    elif row['gender_female']:
        return 'female'
    elif row['gender_other']:
        return 'other'
    else:
        return None

df['gender'] = df.apply(gender, axis=1)

In [58]:
# Columns to include in the consolidation
race_columns = ['american_indian_or_alaska_native', 'asian', 'black_african_american', 'native_hawaiian_other_pacific_islander', 'white', 'unknown']

def consolidate_races(row):
    # Filter the row to include only the specified columns
    selected_row = row[race_columns]
    # Get the column names where the value is 'X'
    races = selected_row.index[selected_row == 'X'].tolist()
    if len(races) > 1:
        return 'Two or more'
    elif len(races) == 1:
        return races[0]
    else:
        return None

# Applying the function to each row to create a new column
df['Race'] = df.apply(consolidate_races, axis=1)

In [79]:
# check ethnicity columns and combine into one column 
columns_to_include = ['hispanic', 'not_hispanic', 'unknown']

# Define the function to apply the logic
def consolidate_ethnicity(row):
    selected_row = row[columns_to_include]  # Select only the specified columns
    ethnicity = selected_row[selected_row == 'X'].index.tolist()  # Get column names where value is 'X'
    if len(ethnicity) > 1:
        return ethnicity
    if len(ethnicity) == 1:
        return ethnicity[0]
    else:
        return None

# Applying the function to each row to create a new column
df['ethnicity'] = df.apply(consolidate_ethnicity, axis=1)

In [81]:
# Rename the birth year column
df.rename(columns={'birth_year_(yyyy)_if_unknown,_please_enter_"9999"': 'birth_year'}, inplace=True)

# Replace '9999' values 
df['birth_year'] = df['birth_year'].replace(9999, pd.NA)

In [82]:
df['birth_year'].unique()

<DatetimeArray>
['1960-02-29 00:00:00', '1973-06-10 00:00:00', '2001-01-26 00:00:00',
 '1985-05-12 00:00:00', '1993-03-20 00:00:00', '1986-06-14 00:00:00',
 '1963-06-20 00:00:00', '2000-11-02 00:00:00', '1984-08-21 00:00:00',
 '1947-02-12 00:00:00',
 ...
 '1958-12-21 00:00:00', '1991-12-13 00:00:00', '1974-12-24 00:00:00',
 '1983-06-13 00:00:00', '1945-12-21 00:00:00', '1973-02-05 00:00:00',
 '1956-01-24 00:00:00', '1989-01-13 00:00:00', '1992-08-07 00:00:00',
 '1978-04-04 00:00:00']
Length: 248, dtype: datetime64[ns]

In [83]:
# looks like the birth year column is being entered as a mm/dd/yyyy. Do we want to rename this to birthdate?

In [85]:
# Combine death date and time columns into a new column
df['death_date_time'] = pd.to_datetime(df['death_date'].astype(str) + ' ' + 
                                       df['death_time'])

In [86]:
# Columns to include in the consolidation
facility_columns = ['municipal_or_county_jail', 'state_prison', 'state_run_boot_camp_prison', 'contracted_boot_camp_prison', 'any_state_or_local_contract_facility', 'other_local_or_state_correctional_facility', 'none_of_the_above']

def consolidate_facility(row):
    # Filter the row to include only the specified columns
    selected_row = row[facility_columns]
    # Get the column names where the value is 'X'
    facility = selected_row.index[selected_row == 'X'].tolist()
    if len(facility) > 1:
        return facility
    elif len(facility) == 1:
        return facility[0]
    else:
        return None

# Applying the function to each row to create a new column
df['death_facility'] = df.apply(consolidate_facility, axis=1)

In [88]:
manner_death_columns = ['execution','accident','use_of_force','homicide','natural_causes','suicide','unavailable_investigation_pending']

def consolidate_manner(row):
    # Filter the row to include only the specified columns
    selected_row = row[manner_death_columns]
    # Get the column names where the value is 'X'
    manner = selected_row.index[selected_row == 'X'].tolist()
    if len(manner) > 1:
        return manner
    elif len(manner) == 1:
        return manner[0]
    else:
        return None

# Applying the function to each row to create a new column
df['manner_of_death'] = df.apply(consolidate_manner, axis=1)

In [91]:
column_names = df.columns.tolist()
print(column_names)

['last_name', 'first_name', 'middle_name', 'gender_male', 'gender_female', 'gender_other', 'gender_other_desc', 'american_indian_or_alaska_native', 'asian', 'black_african_american', 'native_hawaiian_other_pacific_islander', 'white', 'unknown', 'hispanic', 'not_hispanic', 'unknown', 'birth_year', 'death_date', 'death_time', 'death_location', 'death_street_address', 'death_city', 'death_state', 'death_zip', 'municipal_or_county_jail', 'state_prison', 'state_run_boot_camp_prison', 'contracted_boot_camp_prison', 'any_state_or_local_contract_facility', 'other_local_or_state_correctional_facility', 'none_of_the_above', 'agency_name', 'execution', 'accident', 'use_of_force', 'homicide', 'natural_causes', 'suicide', 'unavailable_investigation_pending', 'investigation_pending_desc', 'other', 'other_desc', 'death_description', 'gender', 'Race', 'ethnicity', 'death_date_time', 'death_facility', 'manner_of_death']


In [94]:
df_clean = df[['last_name',
               'first_name',
               'middle_name',
               'gender',
               'gender_other_desc',
               'Race',
               'ethnicity',
               'birth_year',
               'death_date_time',
               'death_facility',
               'death_facility',
               'death_street_address',
               'death_city',
               'death_state',
               'death_zip',
               'manner_of_death',
               'investigation_pending_desc',
               'other_desc',
               'death_description'
              ]]

In [95]:
print(df_clean)

      last_name   first_name   middle_name  gender  gender_other_desc  \
0         Byler        Ricky           Joe    male                NaN   
1    Herrington  Christopher       Michael    male                NaN   
2        Holley      Braylon         Jevon    male                NaN   
3         Drost        Debra        Ashley  female                NaN   
4       Pantoja        JORGE       Ornelas    male                NaN   
..          ...          ...           ...     ...                ...   
246       Verne      Freeman      Emmanuel    male                NaN   
247    Driskill         Dean         Allen    male                NaN   
248      Miller        Caleb             0    male                NaN   
249      Thomas        Ramon  Akeno Warren    male                NaN   
250      Turner        Allen       Maurice    male                NaN   

                       Race            ethnicity birth_year  \
0                     white         not_hispanic 1960-02-29 

In [ ]:
df_clean